# Mesh generation - Gridgen quadtree

Build a **quadtree** unstructured grid with the Gridgen program: start from a coarse base grid and recursively quarter cells (split each cell into four) near the area of interest and along the river network. The result is a **DISV** grid - a discretization by vertices, in which each cell is addressed by a single index instead of a (row, column) pair, so cell sizes can vary across the grid.

Part of the **mesh-generation** series, adapted from the FloPy watershed geoprocessing example (Hughes and others, 2023, *FloPy Workflows for Creating Structured and Unstructured MODFLOW Models*, Groundwater, https://doi.org/10.1111/gwat.13327). Each notebook builds one family of grids over the same synthetic watershed, samples a fine topographic raster onto the grid, and intersects the river network with the grid cells.

- **Rectilinear (DIS + LGR)** (`mf6-mesh-generation-rectilinear`) - structured grids: constant and variable spacing, plus local grid refinement
- **Gridgen quadtree** (`mf6-mesh-generation-gridgen`) - a quadtree unstructured (DISV) grid built with Gridgen *(this notebook)*
- **Triangle & Voronoi** (`mf6-mesh-generation-triangle-voronoi`) - unstructured (DISV) grids built with Triangle and Voronoi

## Imports and setup

Import FloPy and the shared watershed setup from `mf6_mesh_helpers`: the problem extent and contour levels, the fine topographic raster, the boundary/river geometry, and the `resample_topo` / `river_intersection` / `draw_boundary_river` helpers that every mesh-generation notebook uses.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy
import flopy.plot.styles as styles
import matplotlib.pyplot as plt
from flopy.utils.gridgen import Gridgen
from mf6_mesh_helpers import (
    Lx,
    Ly,
    boundary_polygon,
    draw_boundary_river,
    levels,
    resample_topo,
    river_intersection,
    sgs,
    vmax,
    vmin,
)
from mf6_notebook_helpers import set_idomain

## Build the quadtree grid

Start from a coarse (5 km) base grid, then add Gridgen refinement features: a polygon around the area of interest and the river lines. Gridgen writes to a workspace under `models/`.

In [ ]:
gridgen_ws = pl.Path("models/mesh-generation-gridgen")

# area of interest to refine (the LGR child region from the rectilinear notebook)
# exercise: outline the area of interest that Gridgen will refine
# your code here

# a coarse base grid for Gridgen to refine
# exercise: build the coarse base grid Gridgen starts from, with 5,000 m cells
# your code here

gridgen_ws.mkdir(parents=True, exist_ok=True)
# exercise: create the Gridgen object, refine it around the area of interest and the river, and build it
# your code here

# exercise: wrap the result as a FloPy VertexGrid and mark the cells outside the watershed inactive
# your code here

# exercise: sample the land surface onto the new grid and find the cells the river crosses
# your code here


In [ ]:
with styles.USGSMap():
    fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax.set_aspect("equal")
    pmv = flopy.plot.PlotMapView(
        modelgrid=quadtree_grid,
        ax=ax,
    )
    pmv.plot_array(top_qg, ec="0.75", vmin=vmin, vmax=vmax)
    pmv.plot_array(intersection_qg, masked_values=[0], alpha=0.2, cmap="Reds_r")
    pmv.contour_array(top_qg, levels=levels, linewidths=0.3, colors="white")
    pmv.plot_inactive(zorder=100)
    draw_boundary_river(ax)
    ax.set_title("Gridgen quadtree grid")

**What to look for.** The cells are fine (small) near the river and the area of interest and coarse (large) elsewhere - Gridgen quartered cells only where more detail is needed. The colors are land-surface elevation, red shading marks the cells the river crosses, and cells outside the watershed boundary are inactive (white).

**Recap.** Gridgen turns a coarse base grid into a quadtree DISV grid, quartering cells near the area of interest and the river while leaving the rest of the domain coarse. The Triangle and Voronoi notebook builds fully unstructured grids over the same watershed.